# 03 — Simulasi Turnamen Piala Dunia 2026

**Monte Carlo 10.000 iterasi** dari fase grup (72 laga riil) sampai final, memakai bracket resmi FIFA (laga 73–104) termasuk aturan alokasi 8 peringkat-3 terbaik ke slot 32 Besar.

Output disimpan ke `output/predictions.json` dan dipakai dashboard Streamlit.

In [ ]:
import sys
from pathlib import Path

BASE = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(BASE / 'src'))

import numpy as np
import pandas as pd
import plotly.express as px

from simulate import build_sim_inputs, run_simulation, export_predictions, ROUNDS

si = build_sim_inputs()
print('Input simulasi siap: 48 tim, 72 fixture, tabel prediksi semua pasangan knockout.')

Input simulasi siap: 48 tim, 72 fixture, tabel prediksi semua pasangan knockout.


In [ ]:
%%time
agg = run_simulation(si, n_sims=10000, seed=42)
out = export_predictions(si, agg)
total = sum(out['champion'].values())
assert abs(total - 100) < 0.5, f'Total probabilitas juara = {total}, harusnya 100'
print(f"Selesai. Total probabilitas juara = {total:.1f}% (sanity check OK)")

Selesai. Total probabilitas juara = 100.0% (sanity check OK)
CPU times: total: 3 s
Wall time: 2.58 s


## Probabilitas juara — Top 15

In [ ]:
champ = pd.Series(out['champion']).head(15)
fig = px.bar(champ.iloc[::-1], orientation='h', title='Probabilitas Juara Piala Dunia 2026 (%)',
             labels={'index': '', 'value': '%'}).update_layout(showlegend=False)
fig.show()
champ.to_frame('% juara')

,% juara
Argentina,18.23
Spain,13.92
France,8.51
Mexico,8.17
Brazil,6.55
England,5.71
Colombia,5.02
Ecuador,3.86
Canada,3.82
Portugal,3.23


## Probabilitas mencapai tiap babak (Top 12)

In [ ]:
reach_df = pd.DataFrame(out['rounds']).T
reach_df = reach_df.sort_values('Juara', ascending=False).head(12)[ROUNDS]
reach_df

,32 Besar,16 Besar,Perempat Final,Semifinal,Final,Juara
Argentina,96.14,69.39,54.95,39.98,26.99,18.23
Spain,98.17,66.81,46.42,33.51,22.40,13.92
France,93.06,66.16,43.06,27.13,15.43,8.51
Mexico,96.19,68.25,42.71,26.16,14.77,8.17
Brazil,92.54,62.76,39.84,23.65,12.34,6.55
England,94.26,62.75,36.65,22.02,11.45,5.71
Colombia,90.07,59.38,33.96,18.28,9.81,5.02
Ecuador,92.24,58.97,33.05,17.90,9.00,3.86
Canada,95.98,63.51,34.50,16.48,7.93,3.82
Portugal,90.05,53.16,28.77,14.92,6.99,3.23


## Contoh klasemen ekspektasi grup & jalur bracket modal

In [ ]:
print('Grup A (ekspektasi):')
display(pd.DataFrame(out['groups']['A']))
print('\nJalur knockout paling mungkin:')
for rnd in ['Semifinal', 'Final']:
    for m in out['modal_bracket'][rnd]:
        print(f"  {rnd}: {m['home']} vs {m['away']} -> {m['winner']} ({m['p_winner']*100:.0f}%)")

Grup A (ekspektasi):


,tim,exp_poin,p_juara_grup,p_runner_up,p_lolos,elo
0,Mexico,6.64,66.6,22.1,96.2,1980
1,South Korea,4.29,18.4,36.7,74.8,1881
2,Czech Republic,3.51,11.2,27.5,62.5,1802
3,South Africa,2.24,3.7,13.7,32.3,1663



Jalur knockout paling mungkin:
  Semifinal: France vs Spain -> Spain (56%)
  Semifinal: Mexico vs Argentina -> Argentina (58%)
  Final: Spain vs Argentina -> Argentina (52%)
